# Practical 04 — LLM Agents: A Grounded Filing Q&A Pipeline

This notebook runs, end to end and fully offline, the deterministic tools that
power the chapter's filing question-answering **agent**: chunk a small corpus of
SEC-style filings, retrieve the most relevant passages with TF-IDF cosine
similarity, write a *grounded, cited* answer, and grade that answer for
**faithfulness** and **relevance** — looping back when the answer drifts off the
source.

In the real practical you open the folder in **Claude Code** or **Cline** and the
agent (Perceive → Reason → Act) decides *which* tool to call and *interprets* the
outputs. Colab can't drive that agentic loop, so this notebook is the
Colab-friendly counterpart: it calls the same reference tools in `tools/` directly
so you can watch the whole pipeline run on the real bundled data for the fictional
company **NovaCorp Inc.**

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "04-llm-agents"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:  # fresh Colab with nothing cloned yet
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))            # package imports: `from tools.x import ...`
sys.path.insert(0, str(root / "tools"))  # bare imports: `import _common`
os.chdir(root)
print("Practical root:", root)

## Step 0 — Inspect the corpus

The agent is only allowed to answer from what's bundled in `data/corpus/`. Nothing
here touches the network: the "filings" are three short fictional NovaCorp documents.
`tools/_common.py` provides `load_corpus()`, which returns a `{filename: text}` dict.

In [ ]:
import json
from tools._common import load_corpus, tokenize

corpus = load_corpus()
for name, text in corpus.items():
    words = len(text.split())
    print(f"{name:28s}  {words:4d} words")
    print("   ", text.strip().splitlines()[0][:90], "...")
print(f"\nTotal documents: {len(corpus)}")

## Step 1 — Chunk the filings

`tools/chunk.py` splits each document into overlapping fixed-size word windows so a
retriever can score small, comparable passages. Overlap means a fact that straddles
a window boundary still lands intact in at least one chunk. Each chunk carries a
stable id `"<doc>#<index>"`.

In [ ]:
from tools.chunk import chunk_corpus

chunks = chunk_corpus(corpus, size=60, overlap=15)
print(f"{len(chunks)} chunks from {len(corpus)} documents (size=60 words, overlap=15)\n")
for c in chunks[:3]:
    print(f"[{c.id}]  {c.text[:80]}...")

## Step 2 — Retrieve (the `retriever` sub-agent's tool)

`tools/retrieve.py` builds a tiny TF-IDF index over the chunks and ranks them
against a query by cosine similarity — pure standard-library + math, deterministic,
offline. `build_default_retriever()` loads + chunks the bundled corpus for us.
Numbers (like `64` or `2024`) are kept as tokens on purpose: matching figures is how
the pipeline later catches an answer that invents numbers.

In [ ]:
from tools.retrieve import build_default_retriever

retriever = build_default_retriever()
QUESTION = "What is NovaCorp's customer concentration risk?"
hits = retriever.search(QUESTION, k=3)

print(f"Q: {QUESTION}\n")
for c, score in hits:
    print(f"  score={score:.4f}  id={c.id}")
    print(f"      {c.text[:110]}...\n")

The top hit comes from the 10-K risk-factors document — exactly where customer
concentration would be disclosed. These retrieved chunks are the *only* context the
answer is allowed to use.

## Step 3 — Write a grounded, cited answer (the `analyst` sub-agent)

In the agentic project the `analyst` reads the retrieved context and writes an answer
**only** from what it found, citing chunk ids. There is no LLM call in this offline
notebook, so we play the analyst by quoting the retrieved text. We build the answer
from the top chunk so it stays grounded — this is what a faithful analyst produces.

In [ ]:
context_chunks = [c.text for c, _ in hits]
context_ids = [c.id for c, _ in hits]

# A grounded answer: phrased from the retrieved risk-factor language, with citations.
grounded_answer = (
    "NovaCorp's three largest customers together accounted for approximately 48% of "
    "total revenue in fiscal 2024, and the loss of any one would have a material "
    f"adverse effect on results of operations [{context_ids[0]}]."
)
print("Cited chunks:", context_ids)
print("\nAnalyst answer:\n ", grounded_answer)

## Step 4 — Grade faithfulness & relevance (the `grader` sub-agent)

`tools/grade.py` returns two reproducible numbers in [0, 1]:

* **faithfulness** — IDF-weighted share of the *answer* that is supported by the
  retrieved chunks. Distinctive tokens (especially numbers) carry most of the weight,
  so a fabricated figure tanks the score instead of hiding behind common words.
* **relevance** — IDF-weighted share of the *question* covered by the chunks.

A grounded answer should score high on faithfulness; on-topic retrieval should score
high on relevance.

In [ ]:
from tools.grade import grade, faithfulness, relevance

scores = grade(QUESTION, grounded_answer, context_chunks)
print(json.dumps(scores, indent=2))

## Step 5 — The revise loop: what happens when the analyst hallucinates

The agent's loop re-grades and sends the answer back if faithfulness drops. Here we
compare the grounded answer against a *hallucinated* one that invents figures absent
from the context. Watch faithfulness collapse — that's the signal that triggers a
revision.

In [ ]:
hallucinated_answer = (
    "NovaCorp's largest customer accounted for 45% of revenue and net income fell to "
    "12 million dollars after a goodwill writedown in Brazil."
)
# The distinctive tokens '45', '12', 'writedown', 'brazil' never appear in the
# retrieved context, so IDF weighting pushes the hallucination far below threshold.

faith_grounded = faithfulness(grounded_answer, context_chunks)
faith_halluc = faithfulness(hallucinated_answer, context_chunks)
print(f"faithfulness(grounded)      = {faith_grounded:.3f}")
print(f"faithfulness(hallucinated)  = {faith_halluc:.3f}")

THRESHOLD = 0.5
accepted = grounded_answer if faith_grounded >= THRESHOLD else None
print(f"\nGrounded answer accepted? {faith_grounded >= THRESHOLD}")
print(f"Hallucinated answer accepted? {faith_halluc >= THRESHOLD}  "
      "-> the loop would reject it and ask the analyst to revise.")

## Try it — an end-to-end helper, and the "decline" behaviour

Let's wrap retrieve → answer → grade into one function, then mirror the README's
"things to try". First a clean question the filings *do* cover (gross margin), then a
question the filings **don't** cover (net income) where a faithful agent must decline
rather than guess.

In [ ]:
def ask(question, answer, k=3):
    """Retrieve k chunks for `question`, grade `answer` against them, report."""
    hits = retriever.search(question, k=k)
    ctx = [c.text for c, _ in hits]
    s = grade(question, answer, ctx)
    return {
        "question": question,
        "top_chunk": hits[0][0].id,
        "top_score": round(hits[0][1], 4),
        "grade": s,
    }

# Covered: gross margin expanded from 61% to 64% (straight from the MD&A).
print(json.dumps(ask(
    "How did gross margin change year over year?",
    "Gross margin expanded to 64% in fiscal 2024, up from 61% a year earlier.",
), indent=2))

Now the not-answerable case. The corpus reports revenue, gross margin and cash flow,
but **never net income**. The only faithful answer is to decline. We prove the gap by
checking the corpus vocabulary directly: the words a net-income answer needs simply
aren't there for retrieval to ground.

In [ ]:
NOT_COVERED = "What was NovaCorp's net income?"
hits_nc = retriever.search(NOT_COVERED, k=3)
ctx_nc = [c.text for c, _ in hits_nc]

# Build the set of every token present anywhere in the corpus.
corpus_vocab = set()
for text in corpus.values():
    corpus_vocab |= set(tokenize(text))

guessed = "NovaCorp's net income was 150 million dollars."
declined = "Not answerable from the available filings."

print("Is the concept present in the corpus at all?")
for w in ["income", "profit", "150"]:
    print(f"  token {w!r:10s} in corpus vocabulary? {w in corpus_vocab}")

print("\nTop retrieved chunk anyway (id / score):",
      hits_nc[0][0].id, round(hits_nc[0][1], 4))
print("faithfulness(guessed '150 million') =", round(faithfulness(guessed, ctx_nc), 3),
      "(propped up only by generic words like 'million'/'dollars')")
print("\nNo chunk contains a net-income figure, so a faithful agent answers:",
      repr(declined))

## Try it — starving the context with `k=1`

The README suggests dropping `-k` to 1 and watching relevance fall. With fewer
retrieved chunks the context covers less of the question, so relevance drops.

In [ ]:
q = "What supply chain and inventory risks does NovaCorp face from contract manufacturers?"
for k in (1, 2, 3, 5):
    ctx = [c.text for c, _ in retriever.search(q, k=k)]
    print(f"k={k}:  relevance = {relevance(q, ctx):.3f}")
print("\nWith only the top chunk, part of the question is uncovered; widening k to 3 "
      "pulls in the rest of the supply-chain passage and relevance rises.")

## Recap / next steps

We ran the full reference pipeline offline:

1. **load_corpus** — read the bundled NovaCorp filings (`tools/_common.py`).
2. **chunk_corpus** — overlapping word windows with stable ids (`tools/chunk.py`).
3. **Retriever.search** — TF-IDF cosine retrieval of the top-k chunks (`tools/retrieve.py`).
4. *analyst* — write a grounded, cited answer from only the retrieved context.
5. **grade / faithfulness / relevance** — reproducible scores that drive the revise
   loop (`tools/grade.py`).

Key takeaways: a hallucinated figure tanks faithfulness; a question the corpus can't
answer must be declined; and shrinking `k` starves relevance.

To go deeper, open the folder in **Claude Code** or **Cline** and run
`/ask "..."` to see the agent orchestrate these same tools autonomously, or edit the
chunk `size`/`overlap` in `tools/chunk.py` and re-run `python -m pytest -q` to watch
retrieval quality shift.